# 🚀 AS-ViT: Adaptive Subspace Vision Transformers for Multi-Task Learning
### Official Master Execution Notebook for IEEE TPAMI / CVPR
**Author:** Kartikeya Gangwar (Department of Mathematics, University of Delhi)  
**Contact:** kartikeysingh525@protonmail.com | **ORCID:** https://orcid.org/0009-0009-1973-7532  
**Hardware Target:** Free Cloud GPU (Google Colab / Kaggle: Tesla T4, P100, V100, A100)

---

### Overview
This turnkey master notebook provides the complete end-to-end execution pipeline for the **Adaptive Subspace Vision Transformer (AS-ViT)** on the canonical **NYUv2** dense multi-task vision benchmark:
1. **Stage 1 (Feature-Space AMR Discovery):** Autonomous vectorized Gram conflict profiling via `torch.func.vmap`, detecting destructive gradient opposition and cleaving dedicated parameter subspaces.
2. **Stage 2 (Production Retraining):** Clean production training on the discovered topology with fresh AdamW optimizer momentum.
3. **Comparative Multi-Task Benchmarking:** Evaluates all 7 canonical baselines under matched budgets (Single-Task, MT-ViT, PCGrad, CAGrad, MoE-4, MoE-8, AS-ViT).
4. **Diagnostic Metrics:** Generates Tables I-V and Figures 1-5 for IEEE TPAMI submission.


In [ ]:
# Step 1: Cloud GPU Environment & CUDA Acceleration Verification
import os, sys, time, gc
import torch, numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 70)
print(f'[+] PyTorch Version   : {torch.__version__}')
print(f'[+] Hardware Device   : {device}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'[+] Dedicated GPU     : {gpu_name}')
    print(f'[+] Total Cloud VRAM  : {vram_gb:.2f} GB')
    print('[+] CUDA Acceleration : ACTIVE (FP16/AMP enabled)')
else:
    print('[!] WARNING: GPU not detected. Please select T4 GPU in Colab Runtime.')
print('=' * 70)


In [ ]:
# Step 2: Install High-Performance Dependencies & Setup Repo
!pip install --quiet einops h5py pandas matplotlib seaborn scikit-learn pyyaml timm
if not os.path.exists('models') and not os.path.exists('engine'):
    print('[*] Cloning AS-ViT official repository from GitHub...')
    !git clone https://github.com/KartikeyaGangwar/as-vit-multitask.git repo
    %cd repo
    print('[+] Repository cloned and active in working directory!')
else:
    print('[+] Local repository files detected.')


In [ ]:
# Step 3: Fast NYUv2 Multi-Task Dataset Setup (Kaggle / Colab)
import h5py, urllib.request
os.makedirs('data/nyuv2_real', exist_ok=True)
kaggle_nyu_path = '/kaggle/input/nyu-depth-v2/nyu_depth_v2_labeled.mat'
target_mat = 'data/nyuv2_real/nyu_depth_v2_labeled.mat'
if os.path.exists(kaggle_nyu_path):
    print(f'[+] Detected Kaggle mounted NYUv2 dataset at: {kaggle_nyu_path}')
    if not os.path.exists(target_mat):
        os.symlink(kaggle_nyu_path, target_mat)
        print(f'[+] Linked {kaggle_nyu_path} -> {target_mat}')
else:
    print('[*] Setting up cached NYUv2 pipeline...')
    from data.real_nyuv2_pipeline import RealNYUv2Dataset
    train_ds = RealNYUv2Dataset(root='data/nyuv2_real', split='train', download=False)
    val_ds = RealNYUv2Dataset(root='data/nyuv2_real', split='val', download=False)
    print(f'[+] NYUv2 Train Frames: {len(train_ds)} | Test Frames: {len(val_ds)}')


In [ ]:
# Step 4: Run Two-Stage AS-ViT Training (Stage 1 Discovery -> Stage 2 Production)
!python train_as_vit.py --batch_size 8 --disc_epochs 6 --prod_epochs 10


In [ ]:
# Step 5: Run Master Comparative Benchmark Suite (7 Canonical Baselines)
!python -u benchmarks/run_real_nyuv2_benchmark.py --epochs 15 --batch_size 8 --embed_dim 192 --depth 4 --num_heads 4


In [ ]:
# Step 6: Hardware Complexity & Latency Profiling (Table II for TPAMI)
!python benchmarks/benchmark_computational_complexity.py
import json, pandas as pd
with open('results/data/computational_complexity_results.json') as f:
    comp_data = json.load(f)
display(pd.DataFrame(comp_data))


In [ ]:
# Step 7: Layer-Wise Specialization & Zero-Collapse Diagnostics (Table IV for TPAMI)
!python benchmarks/benchmark_architectural_specialization.py
with open('results/data/architectural_specialization_results.json') as f:
    spec_data = json.load(f)
display(pd.DataFrame(spec_data))


In [ ]:
# Step 8: Multi-Dimensional Ablations & Second Benchmark (Tables III & V)
!python benchmarks/run_ablation_experiments.py
!python benchmarks/run_second_benchmark_multitask_vision.py
with open('results/data/second_benchmark_results.json') as f:
    bm2_data = json.load(f)
display(pd.DataFrame(bm2_data))


In [ ]:
# Step 9: Render & Showcase Authentic Real NYUv2 Qualitative Predictions
!python benchmarks/generate_comparative_baseline_versus.py
!python benchmarks/generate_real_nyuv2_qualitative_panel.py
import os, matplotlib.pyplot as plt, matplotlib.image as mpimg
fig_path = "assets/fig5_comparative_baseline_versus.png"
if os.path.exists(fig_path):
    img = mpimg.imread(fig_path)
    plt.figure(figsize=(20, 14), dpi=300)
    plt.imshow(img)
    plt.axis("off")
    plt.title("Head-to-Head Comparative Baseline VERSUS Panel (IEEE TPAMI Fig. 5)", fontsize=15, fontweight="bold", pad=12)
    plt.show()


In [ ]:
# Step 10: Compile LaTeX Manuscript (if manuscript directory present)
import os
if os.path.exists("manuscript/main.tex"):
    !apt-get update -qq && apt-get install -y -qq texlive-latex-base texlive-latex-extra texlive-fonts-recommended
    %cd manuscript
    !pdflatex -interaction=nonstopmode main.tex
    !pdflatex -interaction=nonstopmode main.tex
    %cd ..
    print("[+] IEEE TPAMI Full Paper compiled successfully to: manuscript/main.pdf!")
else:
    print("[*] All benchmark results and figures successfully saved in results/ and assets/.")


In [ ]:
# =============================================================================
# Step 11: Bundle & Download All TPAMI Artifacts (1-Click ZIP)
# =============================================================================
import os
zip_cmd = "zip -r as_vit_tpami_bundle.zip results/ assets/"
if os.path.exists("checkpoints"):
    zip_cmd += " checkpoints/"
if os.path.exists("manuscript/main.pdf"):
    zip_cmd += " manuscript/main.pdf"
os.system(zip_cmd)
print("[+] All benchmark tables, plots, and checkpoints bundled into: as_vit_tpami_bundle.zip")

# If running on Google Colab, trigger automatic browser download:
try:
    from google.colab import files
    files.download("as_vit_tpami_bundle.zip")
    print("[+] Browser download triggered successfully!")
except Exception:
    print("[*] On Kaggle: Download as_vit_tpami_bundle.zip directly from the Output sidebar tab.")
